# Branch Ablation — Which Signal Is Actually Driving the 92.27%?

The full-fusion MLP (LK + RF-DETR + DINOv2 + VideoMAE) hit 92.27% test
accuracy, but its internal validation score hit a perfect 1.0000 during
training -- flagged in `docs/findings.md` (2026-09-19 entry) as an open
question: is the *fusion* doing the work, or would one branch alone get
basically the same number?

This notebook trains the **same MLP architecture** on different column
subsets of whatever `features_*.csv` files exist
(`My Drive/CVPHG/fusion_features/`, produced by
`feature_extraction_fusion.ipynb` with different `BRANCHES` selections)
and reports test accuracy for each, so they're directly comparable.

**No GPU needed for this notebook** -- it's pure sklearn on already-
extracted numeric features, seconds per run regardless of feature
count. (GPU only matters for *producing* the DINOv2/VideoMAE features
in the first place, in `feature_extraction_fusion.ipynb`.)

## Step 0 — Setup

In [ ]:
!pip install scikit-learn -q

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from google.colab import drive

drive.mount('/content/drive')

FEATURES_DIR = '/content/drive/MyDrive/CVPHG/fusion_features'
SEED = 42

## Step 1 — Find and load whatever feature files exist

Looks for any `features_*.csv` in the fusion_features folder. Each
file may cover a different branch subset (e.g. `features_lk_rfdetr.csv`,
`features_dinov2_videomae.csv`) or the full set
(`features_lk_rfdetr_dinov2_videomae.csv`). Merges on `video_id` so a
clip's LK/RF-DETR columns (from one file) and DINOv2/VideoMAE columns
(from another file) can be combined even if they were extracted in
separate runs.

In [ ]:
csv_paths = sorted(glob.glob(os.path.join(FEATURES_DIR, 'features_*.csv')))
print(f"Found {len(csv_paths)} feature file(s):")
for p in csv_paths:
    print(' ', os.path.basename(p))

if not csv_paths:
    raise SystemExit(
        'No features_*.csv found -- run feature_extraction_fusion.ipynb '
        'with at least one BRANCHES selection first.'
    )

In [ ]:
dfs = [pd.read_csv(p) for p in csv_paths]

merged = dfs[0]
for df in dfs[1:]:
    shared_meta = ['video_id', 'label', 'split']
    new_cols = [c for c in df.columns if c not in merged.columns]
    merged = merged.merge(df[['video_id'] + new_cols], on='video_id', how='outer')

print(f"{len(merged)} clips, {len(merged.columns)} total columns after merge")
merged['label'] = merged['label'].fillna(method='ffill').fillna(method='bfill')  # label/split should agree across files for the same video_id
merged = merged.dropna()  # a clip missing from one of the merged files can't be used until that file also covers it
print(f"{len(merged)} clips with complete data across all loaded feature files")

## Step 2 — Define the branch groups and a reusable train/eval function

In [ ]:
BRANCH_GROUPS = {
    'lk_only': lambda cols: [c for c in cols if c.startswith('lk_')],
    'rfdetr_only': lambda cols: [c for c in cols if c.startswith('rfdetr_')],
    'lk_rfdetr': lambda cols: [c for c in cols if c.startswith('lk_') or c.startswith('rfdetr_')],
    'dinov2_only': lambda cols: [c for c in cols if c.startswith('dinov2_')],
    'videomae_only': lambda cols: [c for c in cols if c.startswith('videomae_')],
    'dinov2_videomae': lambda cols: [c for c in cols if c.startswith('dinov2_') or c.startswith('videomae_')],
    'full_fusion': lambda cols: [c for c in cols if c not in ('video_id', 'label', 'split')],
}


def evaluate_branch_group(name, feature_cols, df):
    if not feature_cols:
        return None  # this group's columns aren't present in the loaded data
    X = df[feature_cols].values
    y = (df['label'] == 'riot').astype(int).values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y,
    )
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    mlp = MLPClassifier(
        hidden_layer_sizes=(256, 64), activation='relu', alpha=1e-4,
        early_stopping=True, n_iter_no_change=15, max_iter=500, random_state=SEED,
    )
    mlp.fit(X_train_s, y_train)
    y_pred = mlp.predict(X_test_s)

    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1], zero_division=0)
    return dict(
        branch_group=name, n_features=len(feature_cols), test_accuracy=acc,
        normal_precision=prec[0], normal_recall=rec[0],
        riot_precision=prec[1], riot_recall=rec[1],
        best_val_score=mlp.best_validation_score_,
    )

## Step 3 — Run every branch group that has data available

In [ ]:
results = []
for name, col_fn in BRANCH_GROUPS.items():
    cols = col_fn(merged.columns)
    r = evaluate_branch_group(name, cols, merged)
    if r is not None:
        results.append(r)
        print(f"{name:18s} ({r['n_features']:4d} features): "
              f"acc={r['test_accuracy']:.4f}  val={r['best_val_score']:.4f}")
    else:
        print(f"{name:18s}: skipped (no matching columns in loaded data)")

results_df = pd.DataFrame(results).sort_values('test_accuracy', ascending=False)
results_df

## Step 4 — Save the comparison

In [ ]:
out_path = os.path.join(FEATURES_DIR, 'results', 'branch_ablation.csv')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
results_df.to_csv(out_path, index=False)
print(f"Saved -> {out_path}")

## How to read this

- If `full_fusion` beats every individual/paired group by a real
  margin: fusion is earning its complexity -- report it that way.
- If `dinov2_videomae` (or one of `dinov2_only`/`videomae_only`) is
  close to `full_fusion`'s accuracy on its own: the deep embeddings are
  doing most of the work, and LK+RF-DETR aren't adding much yet -- worth
  saying explicitly rather than crediting the whole architecture.
- If `lk_rfdetr` alone is surprisingly strong: motion + simple counts
  are more informative than expected, and a much cheaper model might
  get most of the way there without the heavy backbones.
- Whatever the outcome, this is the number to cite in `docs/findings.md`
  when explaining what's actually driving the 92.27%, not a guess.